In [38]:
import pandas as pd
import unicodedata

input_path = "ekstraklasa_1989_2023.csv"
output_path = "ekstraklasa_1989_2023_normalized.csv"

df = pd.read_csv(input_path)

def strip_accents(text):
    if pd.isna(text):
        return text
    text = unicodedata.normalize('NFKD', text)
    return ''.join(c for c in text if not unicodedata.combining(c))

for col in df.columns:
    if df[col].dtype == object:
        df[col] = df[col].apply(strip_accents)

rename_map = {
    "data": "date",
    "godzina": "time",
    "dom": "home_team",
    "wyjazd": "away_team",
    "wynik": "score"
}

df = df.rename(columns=rename_map)

df[["home_goals", "away_goals"]] = df["score"].str.split(":", expand=True).astype(int)

df = df.drop(columns=["score", "Unnamed: 0", "time"])

df.to_csv(output_path, index=False)



In [ ]:
2. Wczytywanie danych

In [39]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, log_loss
import matplotlib.pyplot as plt

df = pd.read_csv("ekstraklasa_1989_2023_normalized.csv", parse_dates=["date"])
df = df.sort_values("date").reset_index(drop=True)

df.head()

,date,home_team,away_team,home_goals,away_goals
0,1989-07-29,GKS Katowice,ŁKS Łodz,0,1
1,1989-07-29,Motor Lublin,Widzew Łodz,2,0
2,1989-07-29,Ruch Chorzow,Gornik Zabrze,2,0
3,1989-07-29,Legia Warszawa,FKS Stal Mielec,1,1
4,1989-07-29,Zawisza Bydgoszcz,Slask Wrocław,1,0


In [ ]:
3. Utworzenie targetu (0=H, 1=D, 2=A)

In [40]:
def get_result(row):
    if row.home_goals > row.away_goals:
        return 0
    elif row.home_goals < row.away_goals:
        return 2
    else:
        return 1

df["result"] = df.apply(get_result, axis=1)

In [ ]:
4. Przygotowanie funkcji do rolling features (forma drużyny)
– średnia goli strzelonych
– średnia goli straconych
– średnia punktów (3/1/0)

In [41]:
from collections import deque, defaultdict
def compute_team_form_iterative(df, window=5, min_periods=1, fillna_with=np.nan):
    """
    Dodaje do df kolumny:
      - home_gf_roll, home_ga_roll, home_pts_roll
      - away_gf_roll, away_ga_roll, away_pts_roll

    df musi być posortowane po dacie (rosnąco) i zawierać kolumny:
      'home_team', 'away_team', 'home_goals', 'away_goals'
    """
    # upewnij się o sortowaniu
    df = df.sort_values("date").reset_index(drop=False)  # zachowaj oryginalny index w kolumnie 'index'
    orig_index = df["index"].values  # przywrócimy na końcu

    # struktury historii dla każdej drużyny
    hist_gf = defaultdict(lambda: deque(maxlen=window))
    hist_ga = defaultdict(lambda: deque(maxlen=window))
    hist_pts = defaultdict(lambda: deque(maxlen=window))

    # przygotuj kolumny
    df["home_gf_roll"] = np.nan
    df["home_ga_roll"] = np.nan
    df["home_pts_roll"] = np.nan

    df["away_gf_roll"] = np.nan
    df["away_ga_roll"] = np.nan
    df["away_pts_roll"] = np.nan

    # iteruj po meczach chronologicznie
    for i, row in df.iterrows():
        h = row["home_team"]
        a = row["away_team"]

        # oblicz aktualne rollingi (przed meczem) — średnia z tego, co jest w deque
        def mean_or_nan(dq):
            if len(dq) >= min_periods:
                return float(np.mean(dq))
            else:
                return fillna_with

        # gospodarze (wartości przed doliczeniem dzisiejszego meczu)
        df.at[i, "home_gf_roll"] = mean_or_nan(hist_gf[h])
        df.at[i, "home_ga_roll"] = mean_or_nan(hist_ga[h])
        df.at[i, "home_pts_roll"] = mean_or_nan(hist_pts[h])

        # goście
        df.at[i, "away_gf_roll"] = mean_or_nan(hist_gf[a])
        df.at[i, "away_ga_roll"] = mean_or_nan(hist_ga[a])
        df.at[i, "away_pts_roll"] = mean_or_nan(hist_pts[a])

        # teraz dolicz wynik tego meczu do historii obu drużyn
        try:
            hg = int(row["home_goals"])
            ag = int(row["away_goals"])
        except Exception:
            # jeśli brak/zepsuty wynik — nie dopisuj do historii
            continue

        # punkty
        if hg > ag:
            pts_h, pts_a = 3, 0
        elif hg == ag:
            pts_h, pts_a = 1, 1
        else:
            pts_h, pts_a = 0, 3

        # update history
        hist_gf[h].append(hg); hist_ga[h].append(ag); hist_pts[h].append(pts_h)
        hist_gf[a].append(ag); hist_ga[a].append(hg); hist_pts[a].append(pts_a)

    # przywróć oryginalny index porządkując po kolumnie 'index'
    df = df.set_index("index").loc[orig_index].reset_index(drop=True)
    return df


In [ ]:
5. Obliczanie rolling features i merge z meczami

In [42]:
df = compute_team_form_iterative(df, window=5, min_periods=1, fillna_with=np.nan)

df.tail()


,date,home_team,away_team,home_goals,away_goals,result,home_gf_roll,home_ga_roll,home_pts_roll,away_gf_roll,away_ga_roll,away_pts_roll
9100,2023-10-07,Widzew Łodz,FKS Stal Mielec,1,0,0,1.2,1.8,0.8,2.0,1.6,1.4
9101,2023-10-07,Ruch Chorzow,Pogon Szczecin,0,3,2,1.2,1.8,0.6,3.2,1.4,1.8
9102,2023-10-08,Slask Wrocław,Gornik Zabrze,1,1,1,1.8,0.4,3.0,0.8,1.8,0.8
9103,2023-10-08,KS Cracovia,Jagiellonia Białystok,2,4,2,0.6,1.4,1.0,2.4,1.4,2.0
9104,2023-10-08,Legia Warszawa,Rakow Czestochowa,1,2,2,2.0,1.6,2.0,2.4,1.4,2.4


In [30]:
6. Dodanie systemu ELO 

SyntaxError: invalid syntax (<ipython-input-30-cda7f3a82bb8>, line 1)

In [43]:
def update_elo(elo_a, elo_b, result, k=30):
    # result: 1 = A wins, 0.5 = draw, 0 = A loses
    exp_a = 1 / (1 + 10 ** ((elo_b - elo_a)/400))
    new_a = elo_a + k * (result - exp_a)
    new_b = elo_b + k * ((1-result) - (1-exp_a))
    return new_a, new_b

elos = {}
df["elo_home"] = 0.0
df["elo_away"] = 0.0

for i, row in df.iterrows():
    h, a = row.home_team, row.away_team
    elos.setdefault(h, 1500)
    elos.setdefault(a, 1500)

    df.at[i, "elo_home"] = elos[h]
    df.at[i, "elo_away"] = elos[a]

    if row.result == 0:
        res = 1
    elif row.result == 1:
        res = 0.5
    else:
        res = 0

    new_h, new_a = update_elo(elos[h], elos[a], res)
    elos[h], elos[a] = new_h, new_a

In [ ]:
6a. Różnica zamiast osobnych features (elo_diff, gf_diff, ga_diff, pts_diff)

In [56]:
df["elo_diff"] = df["elo_home"] - df["elo_away"]
df["gf_diff"] = df["home_gf_roll"] - df["away_gf_roll"]
df["ga_diff"] = df["home_ga_roll"] - df["away_ga_roll"]
df["pts_diff"] = df["home_pts_roll"] - df["away_pts_roll"]
df["is_home"]  = 1

In [ ]:
7. Przygotowanie finalnych feature’ów (diff version)

In [57]:
features = [
    "home_gf_roll", "home_ga_roll", "home_pts_roll",
    "away_gf_roll", "away_ga_roll", "away_pts_roll",
    "elo_home", "elo_away"
]

features_diff = [
    "gf_diff", "ga_diff", "pts_diff",
    "elo_diff", "is_home"
]

df_diff = df.drop(columns=features)

df_model = df_diff.dropna(subset=features_diff).copy()

X = df_model[features_diff]
y = df_model["result"]
dates = df_model["date"]
df_model.head()

,date,home_team,away_team,home_goals,away_goals,result,elo_diff,gf_diff,ga_diff,pts_diff,is_home
8,1989-08-02,Slask Wrocław,ŁKS Łodz,1,2,2,-30.0,-1.0,1.0,-3.0,1
9,1989-08-02,Jagiellonia Białystok,Zawisza Bydgoszcz,2,0,0,-15.0,0.0,1.0,-2.0,1
10,1989-08-02,Wisła Krakow,Zagłebie Sosnowiec,1,2,2,0.0,0.0,0.0,0.0,1
11,1989-08-02,Lech Poznan,Olimpia Poznan,1,1,1,-15.0,-1.0,1.0,-1.0,1
12,1989-08-02,Widzew Łodz,Ruch Chorzow,0,3,2,-30.0,-2.0,2.0,-3.0,1


In [ ]:
8. Split czasowy

In [58]:
split_date = dates.quantile(0.8)

X_train = X[dates <= split_date]
y_train = y[dates <= split_date]

X_test = X[dates > split_date]
y_test = y[dates > split_date]

len(X_train), len(X_test)

(7246, 1810)

In [ ]:
#9. Skalowanie + Logistic Regression (multinomial)

In [59]:
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

clf = LogisticRegression(
    multi_class="multinomial",
    solver="lbfgs",
    max_iter=1000
)

clf.fit(X_train_s, y_train)

LogisticRegression(max_iter=1000, multi_class='multinomial')

In [ ]:
#10. Ewaluacja

In [60]:
probs = clf.predict_proba(X_test_s)
preds = clf.predict(X_test_s)

print("Accuracy:", accuracy_score(y_test, preds))
print("LogLoss:", log_loss(y_test, probs))

Accuracy: 0.46574585635359117
LogLoss: 1.051735261834783


In [ ]:
#11. Podgląd predykcji

In [49]:
sample = df_model[df_model.date > split_date].copy()
sample["pred"] = preds
sample[["date", "home_team", "away_team", "result", "pred"]].head(10)

,date,home_team,away_team,result,pred
7292,2017-07-30,Arka Gdynia,Wisła Płock,1,0
7293,2017-07-30,Pogon Szczecin,Jagiellonia Białystok,2,2
7294,2017-07-30,Lech Poznan,Piast Gliwice,0,0
7295,2017-07-31,Korona Kielce,KS Cracovia,0,0
7296,2017-08-04,Piast Gliwice,Slask Wrocław,1,0
7297,2017-08-04,Wisła Płock,Wisła Krakow,2,0
7298,2017-08-05,Zagłebie Lubin,Pogon Szczecin,0,0
7299,2017-08-05,Bruk-Bet Termalica Nieciecza,Legia Warszawa,0,2
7300,2017-08-05,Lechia Gdansk,Gornik Zabrze,1,0
7301,2017-08-06,Jagiellonia Białystok,Sandecja Nowy Sacz,2,0
